Scopus Paper 1 – Leakage-Free Ordinal Deep Learning for Alzheimer Disease Staging from Structural MRI (OASIS-1 experiments)

OASIS-1 data are not included; obtain them from OASIS (https://sites.wustl.edu/oasisbrains/) and accept the OASIS Data Use Agreement. Run the cells in order, top to bottom, with a GPU runtime (Runtime → Change runtime type → T4 GPU).

- **Cells 1–6:** setup, data, labels and model (run every time Colab restarts)
- **Cell 7:** Experiment 1 – leaky (slice-level) vs leakage-free (subject-level) cross-validation
- **Cell 8:** confusion matrix of the standard CNN
- **Cell 9:** Experiment 2 – ordinal CNN with fixed cut-offs
- **Cell 10:** Experiment 3 – ordinal CNN with cut-offs tuned on validation subjects
- **Cell 11:** Experiment 4 – DenseNet121 with the ordinal head
- **Cell 12:** cut-off objective comparison and ordinal metrics (QWK, MAE)
- **Cell 13:** bootstrap confidence intervals and paired statistical tests
- **Cell 14:** Grad-CAM for the final model (trains and saves the final models)
- **Cell 15:** region-level attention analysis
- **Cell 16:** Experiment 6 – augmentation and class-imbalance strategies
Revision analyses (AUCs, occlusion control, repeated cross-validation) and the OASIS-3 external validation are in separate notebooks.

In [ ]:
!pip -q install nibabel openpyxl
# 1. Setup
import os, glob, json, gc, numpy as np, pandas as pd, nibabel as nib, cv2, psutil
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 2. Settings  (change only if you know why)
DATA     = '/content/drive/MyDrive/OASIS1_masked'     # folder with the OASIS-1 .img/.hdr files + the OASIS .xlsx
RESULTS  = '/content/drive/MyDrive/OASIS1_results'    # shared with the ICCIDE notebook (slice cache lives here)
OUT      = f'{RESULTS}/paper1'                        # Paper 1 results are saved here
IMG      = 128      # slice size
N_SLICES = 25       # coronal slices per subject
STEP     = 2        # every 2nd slice
N_FOLDS  = 5
SEED     = 42
BATCH    = 32
EPOCHS   = 60       # maximum epochs (early stopping usually ends sooner)
N_CLS    = 3        # 0 = NonDemented, 1 = Very Mild, 2 = Mild/Moderate
os.makedirs(OUT, exist_ok=True)

In [ ]:
# 3. Labels: keep only MR1 sessions that have a CDR score and an image file
xlsx = [f for f in os.listdir(DATA) if f.endswith(('.xlsx', '.csv'))][0]
df = pd.read_excel(f'{DATA}/{xlsx}') if xlsx.endswith('.xlsx') else pd.read_csv(f'{DATA}/{xlsx}')
df = df[df['ID'].astype(str).str.contains('_MR1') & df['CDR'].notna()].copy()

def find_img(sid):
    hits = glob.glob(f'{DATA}/{sid}_mpr_n*_anon_111_t88_masked_gfc.img')
    return hits[0] if hits else None

df['path']  = df['ID'].map(find_img)
df = df[df['path'].notna()].sort_values('ID').reset_index(drop=True)
df['label'] = (df['CDR'] > 0).astype(int)          # 0 = NonDemented (CDR 0), 1 = Demented (CDR > 0)
ids, y = df['ID'].values, df['label'].values
print(len(df), 'subjects |', (y == 0).sum(), 'NonDemented |', (y == 1).sum(), 'Demented')
print(df['CDR'].value_counts().sort_index().to_string())
assert len(set(ids)) == len(ids), 'duplicate subjects!'
df[['ID', 'Age', 'CDR', 'label']].to_csv(f'{RESULTS}/cohort.csv', index=False)

In [ ]:
# 4. Load / extract coronal slices - memory-safe version (saves progress every 40 subjects; if the session crashes, just run again and it continues)
import gc, psutil

def load_volume(path):
    img = nib.load(path, mmap=False)                           # plain file read (no memory-mapping of Drive files)
    v = np.squeeze(np.asarray(img.dataobj)).astype(np.float32)
    del img
    v /= np.percentile(v[v > 0], 99.5)                         # per-volume intensity normalisation to [0, 1]
    np.clip(v, 0, 1, out=v)
    return v

def coronal_slices(v, centre):
    ap = int(np.argmax(v.shape))                               # the 208-voxel axis is anterior-posterior
    out = []
    for k in range(N_SLICES):
        idx = centre + (k - N_SLICES // 2) * STEP
        sl = np.ascontiguousarray(np.rot90(np.take(v, idx, axis=ap)))   # rows = inferior-superior, cols = left-right
        out.append(cv2.resize(sl, (IMG, IMG), interpolation=cv2.INTER_AREA))
    return np.stack(out).astype(np.float16)

CACHE = f'{RESULTS}/slices_{IMG}_{N_SLICES}x{STEP}.npz'
CHUNK = 40
part_file = lambda s: f'{RESULTS}/slices_part_{s:03d}.npz'

if os.path.exists(CACHE):
    z = np.load(CACHE, allow_pickle=True)
    assert list(z['ids']) == list(ids), 'cache does not match cohort - delete the .npz file and re-run'
    X, CENTRE = z['X'], int(z['centre'])
else:
    cfile = f'{RESULTS}/centre.txt'
    if os.path.exists(cfile):
        CENTRE = int(open(cfile).read())
    else:   # all volumes share one atlas, so one position fits everyone: centre of the brain's front-to-back extent
        cs = []
        for p in df['path'][:10]:
            v = load_volume(p); ap = int(np.argmax(v.shape))
            nzv = np.where(v.sum(axis=tuple(a for a in range(3) if a != ap)) > 0)[0]
            cs.append((nzv[0] + nzv[-1]) // 2); del v
        CENTRE = int(np.median(cs)); open(cfile, 'w').write(str(CENTRE))
    for s in range(0, len(df), CHUNK):
        if os.path.exists(part_file(s)): continue
        part = np.stack([coronal_slices(load_volume(p), CENTRE) for p in df['path'][s:s + CHUNK]])
        np.savez_compressed(part_file(s), X=part, ids=ids[s:s + CHUNK])
        del part; gc.collect()
        print(f'{min(s + CHUNK, len(df))}/{len(df)} subjects done | RAM in use: {psutil.virtual_memory().used / 1e9:.1f} GB')
    parts = [np.load(part_file(s), allow_pickle=True) for s in range(0, len(df), CHUNK)]
    X = np.concatenate([p['X'] for p in parts])
    assert list(np.concatenate([p['ids'] for p in parts])) == list(ids)
    np.savez_compressed(CACHE, X=X, ids=ids, centre=CENTRE)
    for s in range(0, len(df), CHUNK): os.remove(part_file(s))
print('slices:', X.shape, '| centre coronal index:', CENTRE, '| RAM in use: %.1f GB' % (psutil.virtual_memory().used / 1e9))

In [ ]:
# 5. Three-stage labels from CDR
cdr = df['CDR'].astype(float).values
stage = np.select([cdr == 0, cdr == 0.5, cdr >= 1], [0, 1, 2])   # Moderate (CDR 2, n = 2) is merged with Mild
print('3-stage counts [NonDemented, VeryMild, Mild/Moderate]:', np.bincount(stage))
np.save(f'{OUT}/stage3_labels.npy', stage)

In [ ]:
# 6. Model and helper functions
def build_cnn3():
    inp = keras.Input((IMG, IMG, 1))
    x = layers.RandomFlip('horizontal')(inp)          # augmentation: active only during training
    x = layers.RandomRotation(0.03)(x)
    x = layers.RandomTranslation(0.05, 0.05)(x)
    for f in (32, 64, 128):
        x = layers.Conv2D(f, 3, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x); x = layers.Dropout(0.1)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, use_bias=False, kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x); x = layers.Dropout(0.5)(x)
    out = layers.Dense(N_CLS, activation='softmax')(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(3e-4, clipnorm=1.0),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def fit_predict(Xtr, ytr, Xva, yva, Xte):
    keras.backend.clear_session(); tf.random.set_seed(SEED)
    m = build_cnn3()
    cw = compute_class_weight('balanced', classes=np.arange(N_CLS), y=ytr)
    cb = [keras.callbacks.EarlyStopping('val_loss', patience=10, restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=4, min_lr=1e-6)]
    m.fit(Xtr[..., None].astype('float32'), ytr,
          validation_data=(Xva[..., None].astype('float32'), yva),
          epochs=EPOCHS, batch_size=BATCH, class_weight=dict(enumerate(cw)), callbacks=cb, verbose=0)
    return m.predict(Xte[..., None].astype('float32'), batch_size=256, verbose=0)

def scores(y, p):
    return dict(acc=accuracy_score(y, p), bal_acc=balanced_accuracy_score(y, p),
                macro_f1=f1_score(y, p, average='macro'))
print('Model ready:', build_cnn3().count_params(), 'parameters')

In [ ]:
# 7. Experiment 1: leakage-free (subject-level) vs leaky (slice-level) 5-fold CV
#    Finished folds are saved; if Colab disconnects, run cells 1-6 again and then this cell - it continues.
NS = X.shape[1]
E1 = f'{OUT}/exp1'; os.makedirs(E1, exist_ok=True)
rows = []

# ---- A) Leakage-free: split by SUBJECT ----
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
for k, (tr, te) in enumerate(skf.split(np.zeros(len(stage)), stage)):
    f = f'{E1}/subject_fold{k}.json'
    if os.path.exists(f):
        rows.append(json.load(open(f))); print('subject-level fold', k, 'loaded'); continue
    tr, va = train_test_split(tr, test_size=0.15, stratify=stage[tr], random_state=SEED)
    expand = lambda idx: (X[idx].reshape(-1, IMG, IMG), np.repeat(stage[idx], NS))
    (Xtr, ytr), (Xva, yva), (Xte, yte) = expand(tr), expand(va), expand(te)
    P = fit_predict(Xtr, ytr, Xva, yva, Xte)
    subjP = P.reshape(len(te), NS, N_CLS).mean(1)       # average 25 slices -> 1 decision per subject
    r = {'setup': 'subject-level (leakage-free)', 'fold': k,
         **{f'slice_{a}': float(v) for a, v in scores(yte, P.argmax(1)).items()},
         **{f'subject_{a}': float(v) for a, v in scores(stage[te], subjP.argmax(1)).items()},
         'test_idx': te.tolist(), 'probs': subjP.tolist()}
    json.dump(r, open(f, 'w')); rows.append(r)
    print('subject-level fold', k, {a: round(v, 3) for a, v in r.items() if a.startswith(('slice_', 'subject_'))})

# ---- B) Leaky: split by SLICE ----
Xs, ys = X.reshape(-1, IMG, IMG), np.repeat(stage, NS)
skf2 = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
for k, (tr, te) in enumerate(skf2.split(np.zeros(len(ys)), ys)):
    f = f'{E1}/leaky_fold{k}.json'
    if os.path.exists(f):
        rows.append(json.load(open(f))); print('leaky fold', k, 'loaded'); continue
    tr, va = train_test_split(tr, test_size=0.15, stratify=ys[tr], random_state=SEED)
    P = fit_predict(Xs[tr], ys[tr], Xs[va], ys[va], Xs[te])
    r = {'setup': 'slice-level (leaky)', 'fold': k,
         **{f'slice_{a}': float(v) for a, v in scores(ys[te], P.argmax(1)).items()}}
    json.dump(r, open(f, 'w')); rows.append(r)
    print('leaky fold', k, {a: round(v, 3) for a, v in r.items() if a.startswith('slice_')})

# ---- Summary ----
res = pd.DataFrame(rows).drop(columns=['test_idx', 'probs'], errors='ignore')
summary = res.drop(columns='fold').groupby('setup').agg(['mean', 'std']).round(3)
summary.to_csv(f'{E1}/summary.csv')
print('\n===== SUMMARY (3 stages, Custom CNN, 5-fold) =====')
print(summary.T)

In [ ]:
# 8. Where does the model make mistakes? (leakage-free, subject level, all 5 folds pooled)
from sklearn.metrics import confusion_matrix, classification_report
E1 = f'{OUT}/exp1'
y_true, y_pred = [], []
for k in range(N_FOLDS):
    r = json.load(open(f'{E1}/subject_fold{k}.json'))
    y_true += [int(stage[i]) for i in r['test_idx']]
    y_pred += list(np.argmax(r['probs'], axis=1))
names = ['NonDemented', 'VeryMild', 'Mild/Moderate']
print('Majority-class baseline accuracy:', round(np.mean(np.array(y_true) == 0), 3))
print('\nConfusion matrix (rows = true, columns = predicted):')
print(pd.DataFrame(confusion_matrix(y_true, y_pred), index=names, columns=names))
print('\n', classification_report(y_true, y_pred, target_names=names, digits=3))

In [ ]:
# 9. Experiment 2: ordinal Custom CNN (stages are ordered: NonDemented < VeryMild < Mild/Moderate)
from sklearn.metrics import confusion_matrix, classification_report
E2 = f'{OUT}/exp2_ordinal'; os.makedirs(E2, exist_ok=True)
NS = X.shape[1]

def build_cnn_ordinal():
    inp = keras.Input((IMG, IMG, 1))
    x = layers.RandomFlip('horizontal')(inp)
    x = layers.RandomRotation(0.03)(x)
    x = layers.RandomTranslation(0.05, 0.05)(x)
    for f in (32, 64, 128):
        x = layers.Conv2D(f, 3, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x); x = layers.Dropout(0.1)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, use_bias=False, kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x); x = layers.Dropout(0.5)(x)
    out = layers.Dense(N_CLS - 1, activation='sigmoid')(x)      # [P(stage > 0), P(stage > 1)]
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(3e-4, clipnorm=1.0), loss='binary_crossentropy')
    return m

to_ord = lambda y: np.stack([y > 0, y > 1], axis=1).astype('float32')
ord_to_stage = lambda p: (p > 0.5).sum(axis=1)

rows2 = []
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)      # same subject folds as Experiment 1
for k, (tr, te) in enumerate(skf.split(np.zeros(len(stage)), stage)):
    f = f'{E2}/fold{k}.json'
    if os.path.exists(f):
        rows2.append(json.load(open(f))); print('ordinal fold', k, 'loaded'); continue
    tr, va = train_test_split(tr, test_size=0.15, stratify=stage[tr], random_state=SEED)
    expand = lambda idx: (X[idx].reshape(-1, IMG, IMG)[..., None].astype('float32'), np.repeat(stage[idx], NS))
    (Xtr, ytr), (Xva, yva), (Xte, yte) = expand(tr), expand(va), expand(te)
    cw = compute_class_weight('balanced', classes=np.arange(N_CLS), y=ytr)
    keras.backend.clear_session(); tf.random.set_seed(SEED)
    m = build_cnn_ordinal()
    cb = [keras.callbacks.EarlyStopping('val_loss', patience=10, restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=4, min_lr=1e-6)]
    m.fit(Xtr, to_ord(ytr), sample_weight=cw[ytr], validation_data=(Xva, to_ord(yva), cw[yva]),
          epochs=EPOCHS, batch_size=BATCH, callbacks=cb, verbose=0)
    P = m.predict(Xte, batch_size=256, verbose=0)
    subjP = P.reshape(len(te), NS, N_CLS - 1).mean(1)              # average 25 slices per subject
    pred = ord_to_stage(subjP)
    r = {'fold': k, **{f'subject_{a}': float(v) for a, v in scores(stage[te], pred).items()},
         'test_idx': te.tolist(), 'pred': pred.tolist()}
    json.dump(r, open(f, 'w')); rows2.append(r)
    print('ordinal fold', k, {a: round(v, 3) for a, v in r.items() if a.startswith('subject_')})

res2 = pd.DataFrame(rows2)[['subject_acc', 'subject_bal_acc', 'subject_macro_f1']]
print('\n===== ORDINAL Custom CNN (leakage-free, 5-fold) =====')
print(res2.agg(['mean', 'std']).round(3))

yt = [int(stage[i]) for r in rows2 for i in r['test_idx']]
yp = [p for r in rows2 for p in r['pred']]
names = ['NonDemented', 'VeryMild', 'Mild/Moderate']
print('\nConfusion matrix (rows = true, columns = predicted):')
print(pd.DataFrame(confusion_matrix(yt, yp), index=names, columns=names))
print('\n', classification_report(yt, yp, target_names=names, digits=3))

In [ ]:
# 10. Experiment 3: ordinal Custom CNN + cut-offs tuned on validation subjects (no test leakage)
E3 = f'{OUT}/exp3_ordinal_tuned'; os.makedirs(E3, exist_ok=True)

def subject_score(model, idx):
    """Predicted disease score per subject (0 to 2) = average over 25 slices of P(stage>0) + P(stage>1)."""
    Xi = X[idx].reshape(-1, IMG, IMG)[..., None].astype('float32')
    P = model.predict(Xi, batch_size=256, verbose=0)
    return P.reshape(len(idx), NS, N_CLS - 1).sum(axis=2).mean(axis=1)

def apply_cuts(s, t1, t2):
    return np.where(s < t1, 0, np.where(s < t2, 1, 2))

def tune_cuts(s, y):
    best, grid = (0.5, 1.5, -1), np.arange(0.10, 1.95, 0.05)
    for t1 in grid:
        for t2 in grid[grid > t1]:
            b = balanced_accuracy_score(y, apply_cuts(s, t1, t2))
            if b > best[2]: best = (t1, t2, b)
    return best[:2]

rows3 = []
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)      # same subject folds as before
for k, (tr, te) in enumerate(skf.split(np.zeros(len(stage)), stage)):
    f = f'{E3}/fold{k}.json'
    if os.path.exists(f):
        rows3.append(json.load(open(f))); print('tuned fold', k, 'loaded'); continue
    tr, va = train_test_split(tr, test_size=0.15, stratify=stage[tr], random_state=SEED)
    expand = lambda idx: (X[idx].reshape(-1, IMG, IMG)[..., None].astype('float32'), np.repeat(stage[idx], NS))
    (Xtr, ytr), (Xva, yva) = expand(tr), expand(va)
    cw = compute_class_weight('balanced', classes=np.arange(N_CLS), y=ytr)
    keras.backend.clear_session(); tf.random.set_seed(SEED)
    m = build_cnn_ordinal()
    cb = [keras.callbacks.EarlyStopping('val_loss', patience=10, restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=4, min_lr=1e-6)]
    m.fit(Xtr, to_ord(ytr), sample_weight=cw[ytr], validation_data=(Xva, to_ord(yva), cw[yva]),
          epochs=EPOCHS, batch_size=BATCH, callbacks=cb, verbose=0)
    s_va, s_te = subject_score(m, va), subject_score(m, te)
    t1, t2 = tune_cuts(s_va, stage[va])                             # cut-offs chosen on VALIDATION subjects only
    pred = apply_cuts(s_te, t1, t2)
    r = {'fold': k, 't1': float(t1), 't2': float(t2),
         **{f'subject_{a}': float(v) for a, v in scores(stage[te], pred).items()},
         'test_idx': te.tolist(), 'score': s_te.tolist(), 'pred': pred.tolist(),
         'val_idx': va.tolist(), 'val_score': s_va.tolist()}
    json.dump(r, open(f, 'w')); rows3.append(r)
    print(f'tuned fold {k} | cut-offs {t1:.2f}, {t2:.2f} |',
          {a: round(v, 3) for a, v in r.items() if a.startswith('subject_')})

res3 = pd.DataFrame(rows3)[['subject_acc', 'subject_bal_acc', 'subject_macro_f1']]
print('\n===== ORDINAL + TUNED CUT-OFFS (leakage-free, 5-fold) =====')
print(res3.agg(['mean', 'std']).round(3))

yt = [int(stage[i]) for r in rows3 for i in r['test_idx']]
yp = [p for r in rows3 for p in r['pred']]
names = ['NonDemented', 'VeryMild', 'Mild/Moderate']
print('\nConfusion matrix (rows = true, columns = predicted):')
print(pd.DataFrame(confusion_matrix(yt, yp), index=names, columns=names))
print('\n', classification_report(yt, yp, target_names=names, digits=3))

In [ ]:
# 11. Experiment 4: DenseNet121 (ImageNet) + ordinal head + cut-offs tuned on validation subjects
E4 = f'{OUT}/exp4_densenet_ordinal'; os.makedirs(E4, exist_ok=True)
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def build_densenet_ordinal():
    inp = keras.Input((IMG, IMG, 1))
    x = layers.RandomFlip('horizontal')(inp)
    x = layers.RandomRotation(0.03)(x)
    x = layers.RandomTranslation(0.05, 0.05)(x)
    x = layers.Resizing(224, 224)(x)
    x = layers.Concatenate()([x, x, x])                                     # grey -> 3 channels
    x = layers.Normalization(mean=MEAN, variance=[s ** 2 for s in STD])(x)  # ImageNet normalisation
    base = keras.applications.DenseNet121(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
    base.trainable = False
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x); x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x); x = layers.Dropout(0.3)(x)
    out = layers.Dense(N_CLS - 1, activation='sigmoid')(x)                  # [P(stage > 0), P(stage > 1)]
    return keras.Model(inp, out), base

rows4 = []
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)      # same subject folds as before
for k, (tr, te) in enumerate(skf.split(np.zeros(len(stage)), stage)):
    f = f'{E4}/fold{k}.json'
    if os.path.exists(f):
        rows4.append(json.load(open(f))); print('densenet fold', k, 'loaded'); continue
    tr, va = train_test_split(tr, test_size=0.15, stratify=stage[tr], random_state=SEED)
    expand = lambda idx: (X[idx].reshape(-1, IMG, IMG)[..., None].astype('float32'), np.repeat(stage[idx], NS))
    (Xtr, ytr), (Xva, yva) = expand(tr), expand(va)
    cw = compute_class_weight('balanced', classes=np.arange(N_CLS), y=ytr)
    keras.backend.clear_session(); tf.random.set_seed(SEED)
    m, base = build_densenet_ordinal()
    cb = lambda: [keras.callbacks.EarlyStopping('val_loss', patience=6, restore_best_weights=True),
                  keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=3, min_lr=1e-7)]
    # Phase 1: train the head only (backbone frozen)
    m.compile(optimizer=keras.optimizers.Adam(1e-4, clipnorm=1.0), loss='binary_crossentropy')
    m.fit(Xtr, to_ord(ytr), sample_weight=cw[ytr], validation_data=(Xva, to_ord(yva), cw[yva]),
          epochs=20, batch_size=BATCH, callbacks=cb(), verbose=0)
    # Phase 2: fine-tune the last 30 backbone layers (BatchNorm layers stay frozen)
    for layer in base.layers[-30:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True
    m.compile(optimizer=keras.optimizers.Adam(1e-5, clipnorm=1.0), loss='binary_crossentropy')
    m.fit(Xtr, to_ord(ytr), sample_weight=cw[ytr], validation_data=(Xva, to_ord(yva), cw[yva]),
          epochs=20, batch_size=BATCH, callbacks=cb(), verbose=0)
    s_va, s_te = subject_score(m, va), subject_score(m, te)
    t1, t2 = tune_cuts(s_va, stage[va])                              # cut-offs from VALIDATION subjects only
    pred = apply_cuts(s_te, t1, t2)
    r = {'fold': k, 't1': float(t1), 't2': float(t2),
         **{f'subject_{a}': float(v) for a, v in scores(stage[te], pred).items()},
         'test_idx': te.tolist(), 'score': s_te.tolist(), 'pred': pred.tolist(),
         'val_idx': va.tolist(), 'val_score': s_va.tolist()}
    json.dump(r, open(f, 'w')); rows4.append(r)
    print(f'densenet fold {k} | cut-offs {t1:.2f}, {t2:.2f} |',
          {a: round(v, 3) for a, v in r.items() if a.startswith('subject_')})

res4 = pd.DataFrame(rows4)[['subject_acc', 'subject_bal_acc', 'subject_macro_f1']]
print('\n===== DenseNet121 + ORDINAL + TUNED CUT-OFFS (leakage-free, 5-fold) =====')
print(res4.agg(['mean', 'std']).round(3))

yt = [int(stage[i]) for r in rows4 for i in r['test_idx']]
yp = [p for r in rows4 for p in r['pred']]
names = ['NonDemented', 'VeryMild', 'Mild/Moderate']
print('\nConfusion matrix (rows = true, columns = predicted):')
print(pd.DataFrame(confusion_matrix(yt, yp), index=names, columns=names))
print('\n', classification_report(yt, yp, target_names=names, digits=3))

In [ ]:
# 12. Analysis (no training): cut-off objective comparison + ordinal metrics, using saved scores
from sklearn.metrics import cohen_kappa_score, mean_absolute_error, f1_score
E3 = f'{OUT}/exp3_ordinal_tuned'
E4 = f'{OUT}/exp4_densenet_ordinal'

def apply_cuts(s, t1, t2):
    return np.where(s < t1, 0, np.where(s < t2, 1, 2))

def tune(s, y, metric):
    grid, best = np.arange(0.10, 1.95, 0.05), (0.5, 1.5, -1)
    for t1 in grid:
        for t2 in grid[grid > t1]:
            v = metric(y, apply_cuts(s, t1, t2))
            if v > best[2]: best = (t1, t2, v)
    return best[:2]

mf1 = lambda y, p: f1_score(y, p, average='macro')
qwk = lambda y, p: cohen_kappa_score(y, p, weights='quadratic')

rows = []
for model_name, E in [('Custom CNN ordinal', E3), ('DenseNet121 ordinal', E4)]:
    for obj_name, obj in [('balanced accuracy', balanced_accuracy_score), ('macro F1', mf1)]:
        yt, yp = [], []
        for k in range(N_FOLDS):
            r = json.load(open(f'{E}/fold{k}.json'))
            t1, t2 = tune(np.array(r['val_score']), stage[r['val_idx']], obj)   # cut-offs from VALIDATION only
            yt += list(stage[r['test_idx']]); yp += list(apply_cuts(np.array(r['score']), t1, t2))
        yt, yp = np.array(yt), np.array(yp)
        cm = confusion_matrix(yt, yp, labels=[0, 1, 2])
        rec = cm.diagonal() / cm.sum(axis=1)
        rows.append({'model': model_name, 'cut-offs tuned for': obj_name,
                     'acc': accuracy_score(yt, yp), 'bal_acc': balanced_accuracy_score(yt, yp),
                     'macro_F1': mf1(yt, yp), 'QWK': qwk(yt, yp), 'MAE': mean_absolute_error(yt, yp),
                     'ND_recall': rec[0], 'VeryMild_recall': rec[1], 'MildMod_recall': rec[2]})

table = pd.DataFrame(rows).round(3)
table.to_csv(f'{OUT}/analysis_cutoff_objectives.csv', index=False)
pd.set_option('display.width', 200)
print(table.to_string(index=False))

In [ ]:
# 13. Statistics: bootstrap 95% CIs and paired comparisons (no training)
from sklearn.metrics import cohen_kappa_score, f1_score
from scipy.stats import binomtest
E1, E3, E4 = f'{OUT}/exp1', f'{OUT}/exp3_ordinal_tuned', f'{OUT}/exp4_densenet_ordinal'
rng = np.random.default_rng(SEED)

def apply_cuts(s, t1, t2):
    return np.where(s < t1, 0, np.where(s < t2, 1, 2))

def tune(s, y, metric):
    grid, best = np.arange(0.10, 1.95, 0.05), (0.5, 1.5, -1)
    for t1 in grid:
        for t2 in grid[grid > t1]:
            v = metric(y, apply_cuts(s, t1, t2))
            if v > best[2]: best = (t1, t2, v)
    return best[:2]

mf1 = lambda y, p: f1_score(y, p, average='macro', zero_division=0)
qwk = lambda y, p: cohen_kappa_score(y, p, weights='quadratic')

def preds_exp1():
    p = np.full(len(stage), -1)
    for k in range(N_FOLDS):
        r = json.load(open(f'{E1}/subject_fold{k}.json'))
        p[r['test_idx']] = np.argmax(r['probs'], axis=1)
    return p

def preds_tuned(E, obj):
    p = np.full(len(stage), -1)
    for k in range(N_FOLDS):
        r = json.load(open(f'{E}/fold{k}.json'))
        t1, t2 = tune(np.array(r['val_score']), stage[r['val_idx']], obj)   # cut-offs from VALIDATION only
        p[r['test_idx']] = apply_cuts(np.array(r['score']), t1, t2)
    return p

models = {
    'Exp1 standard CNN':          preds_exp1(),
    'Custom CNN ordinal (final)': preds_tuned(E3, mf1),
    'DenseNet121 ordinal':        preds_tuned(E4, balanced_accuracy_score),
}
y = stage
metrics = {'bal_acc': balanced_accuracy_score, 'macro_F1': mf1, 'QWK': qwk}

# ---- 95% bootstrap confidence intervals (2,000 resamples of subjects) ----
B = 2000
boot_idx = [rng.integers(0, len(y), len(y)) for _ in range(B)]
rows = []
for name, p in models.items():
    row = {'model': name}
    for mname, fn in metrics.items():
        vals = [fn(y[i], p[i]) for i in boot_idx]
        row[mname] = f'{fn(y, p):.3f} [{np.percentile(vals, 2.5):.3f}, {np.percentile(vals, 97.5):.3f}]'
    rows.append(row)
print('===== Point estimate [95% bootstrap CI] =====')
print(pd.DataFrame(rows).to_string(index=False))

# ---- Paired comparisons: final model vs others ----
final = models['Custom CNN ordinal (final)']
print('\n===== Paired comparisons with the final model =====')
for name in ['Exp1 standard CNN', 'DenseNet121 ordinal']:
    other = models[name]
    diffs = np.array([balanced_accuracy_score(y[i], final[i]) - balanced_accuracy_score(y[i], other[i]) for i in boot_idx])
    d = balanced_accuracy_score(y, final) - balanced_accuracy_score(y, other)
    p_boot = min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))
    c_final, c_other = final == y, other == y
    b, c = int(np.sum(c_final & ~c_other)), int(np.sum(~c_final & c_other))
    p_mcn = binomtest(b, b + c, 0.5).pvalue if b + c > 0 else 1.0
    print(f'Final vs {name}: balanced-accuracy difference {d:+.3f} '
          f'[95% CI {np.percentile(diffs, 2.5):+.3f}, {np.percentile(diffs, 97.5):+.3f}], bootstrap p = {p_boot:.3f} | '
          f'McNemar: final-only correct {b}, other-only correct {c}, p = {p_mcn:.3f}')

pd.DataFrame(rows).to_csv(f'{OUT}/statistics_bootstrap.csv', index=False)

In [ ]:
# 14. Grad-CAM per stage for the FINAL model (ordinal Custom CNN, cut-offs tuned for macro F1)
#     Trains the 5 folds once and SAVES the models; on re-runs the saved models are loaded.
import scipy.ndimage as ndi
from scipy.stats import kruskal
from sklearn.metrics import f1_score, cohen_kappa_score
E5 = f'{OUT}/exp5_gradcam'; os.makedirs(E5, exist_ok=True)
NS = X.shape[1]
names = ['NonDemented', 'VeryMild', 'Mild/Moderate']

def build_cnn_ordinal():
    inp = keras.Input((IMG, IMG, 1))
    x = layers.RandomFlip('horizontal')(inp)
    x = layers.RandomRotation(0.03)(x)
    x = layers.RandomTranslation(0.05, 0.05)(x)
    for f in (32, 64, 128):
        x = layers.Conv2D(f, 3, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x); x = layers.Dropout(0.1)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, use_bias=False, kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x); x = layers.Dropout(0.5)(x)
    out = layers.Dense(N_CLS - 1, activation='sigmoid')(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(3e-4, clipnorm=1.0), loss='binary_crossentropy')
    return m

to_ord = lambda y: np.stack([y > 0, y > 1], axis=1).astype('float32')
mf1 = lambda y, p: f1_score(y, p, average='macro', zero_division=0)

def apply_cuts(s, t1, t2):
    return np.where(s < t1, 0, np.where(s < t2, 1, 2))

def tune(s, y, metric):
    grid, best = np.arange(0.10, 1.95, 0.05), (0.5, 1.5, -1)
    for t1 in grid:
        for t2 in grid[grid > t1]:
            v = metric(y, apply_cuts(s, t1, t2))
            if v > best[2]: best = (t1, t2, v)
    return best[:2]

def subject_score(model, idx):
    Xi = X[idx].reshape(-1, IMG, IMG)[..., None].astype('float32')
    P = model.predict(Xi, batch_size=256, verbose=0)
    return P.reshape(len(idx), NS, N_CLS - 1).sum(axis=2).mean(axis=1)

def brain_mask(sl):
    m = ndi.binary_fill_holes(ndi.binary_closing(sl > 0.10, iterations=2))
    lab, n = ndi.label(m)
    return lab == (np.argmax(np.bincount(lab.ravel())[1:]) + 1) if n > 1 else m

def gradcam_subject(grad_model, sidx, sign):
    """Grad-CAM for the 25 slices of one subject. sign=+1 explains 'more disease', -1 explains 'healthy'."""
    xs = tf.convert_to_tensor(X[sidx][..., None].astype('float32'))
    with tf.GradientTape() as tape:
        A, out = grad_model(xs, training=False)
        target = sign * tf.reduce_sum(out, axis=1)                # disease score = P(>0) + P(>1)
    g = tape.gradient(target, A)
    w = tf.reduce_mean(g, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(w * A, axis=-1)).numpy()
    cam = np.stack([cv2.resize(c, (IMG, IMG)) for c in cam])
    return cam / (cam.max(axis=(1, 2), keepdims=True) + 1e-8)

inbrain = np.full(len(stage), np.nan)
pred_all = np.full(len(stage), -1)
cam_sum = {s: np.zeros((NS, IMG, IMG)) for s in range(N_CLS)}
img_sum = {s: np.zeros((NS, IMG, IMG)) for s in range(N_CLS)}
n_ok = {s: 0 for s in range(N_CLS)}

skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)       # same subject folds as before
for k, (tr, te) in enumerate(skf.split(np.zeros(len(stage)), stage)):
    tr, va = train_test_split(tr, test_size=0.15, stratify=stage[tr], random_state=SEED)
    mfile = f'{E5}/model_fold{k}.keras'
    if os.path.exists(mfile):
        m = keras.models.load_model(mfile); print(f'fold {k}: model loaded')
    else:
        expand = lambda idx: (X[idx].reshape(-1, IMG, IMG)[..., None].astype('float32'), np.repeat(stage[idx], NS))
        (Xtr, ytr), (Xva, yva) = expand(tr), expand(va)
        cw = compute_class_weight('balanced', classes=np.arange(N_CLS), y=ytr)
        keras.backend.clear_session(); tf.random.set_seed(SEED)
        m = build_cnn_ordinal()
        cb = [keras.callbacks.EarlyStopping('val_loss', patience=10, restore_best_weights=True),
              keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=4, min_lr=1e-6)]
        m.fit(Xtr, to_ord(ytr), sample_weight=cw[ytr], validation_data=(Xva, to_ord(yva), cw[yva]),
              epochs=EPOCHS, batch_size=BATCH, callbacks=cb, verbose=0)
        m.save(mfile); print(f'fold {k}: model trained and saved')
        del Xtr, Xva

    t1, t2 = tune(subject_score(m, va), stage[va], mf1)            # cut-offs from VALIDATION only (macro F1)
    pred = apply_cuts(subject_score(m, te), t1, t2)
    pred_all[te] = pred

    conv = [l for l in m.layers if isinstance(l, layers.ReLU) and len(l.output.shape) == 4][-1]
    grad_model = keras.Model(m.inputs, [conv.output, m.output])
    for i, sidx in enumerate(te):
        cam = gradcam_subject(grad_model, sidx, 1 if pred[i] > 0 else -1)
        fr = [(c * brain_mask(X[sidx, j].astype('float32'))).sum() / (c.sum() + 1e-8) for j, c in enumerate(cam)]
        inbrain[sidx] = np.mean(fr)
        if pred[i] == stage[sidx]:                                  # average maps of CORRECT predictions per stage
            cam_sum[stage[sidx]] += cam; img_sum[stage[sidx]] += X[sidx].astype('float32'); n_ok[stage[sidx]] += 1
    print(f'fold {k}: cut-offs {t1:.2f}, {t2:.2f} | Grad-CAM done for {len(te)} test subjects')

# ---- Results ----
print('\n===== Final model, this run (leakage-free, 5-fold) =====')
print(f'acc {accuracy_score(stage, pred_all):.3f} | bal_acc {balanced_accuracy_score(stage, pred_all):.3f} | '
      f'macro_F1 {mf1(stage, pred_all):.3f} | QWK {cohen_kappa_score(stage, pred_all, weights="quadratic"):.3f}')

res = pd.DataFrame({'ID': ids, 'true': stage, 'pred': pred_all, 'inbrain': inbrain})
res.to_csv(f'{E5}/gradcam_inbrain.csv', index=False)
print('\n===== Fraction of Grad-CAM attention inside brain tissue =====')
for col in ['true', 'pred']:
    t = res.groupby(col)['inbrain'].agg(['mean', 'std', 'count']).round(3)
    t.index = [names[i] for i in t.index]
    print(f'\nBy {col} stage:'); print(t)
print('\nOverall: %.3f +/- %.3f' % (res.inbrain.mean(), res.inbrain.std()))
print('Kruskal-Wallis across true stages: p = %.4f' % kruskal(*[res.inbrain[res.true == s] for s in range(N_CLS)]).pvalue)
print('Correctly classified subjects used for the average maps:', {names[s]: n_ok[s] for s in range(N_CLS)})

# ---- Figure: average Grad-CAM of correctly classified subjects, per stage, at 3 slice positions ----
cols = [6, 12, 18]
fig, ax = plt.subplots(N_CLS, len(cols), figsize=(3 * len(cols), 3 * N_CLS))
for s in range(N_CLS):
    for c, j in enumerate(cols):
        a = ax[s, c]; a.axis('off')
        if n_ok[s] == 0: continue
        a.imshow(img_sum[s][j] / n_ok[s], cmap='gray')
        a.imshow(cam_sum[s][j] / n_ok[s], cmap='jet', alpha=0.4)
        if c == 0: a.set_title(f'{names[s]} (n = {n_ok[s]})', loc='left', fontsize=10)
        else: a.set_title(f'slice {j + 1} of {NS}', fontsize=9)
plt.suptitle('Average Grad-CAM of correctly classified subjects per stage (final model)', fontsize=11)
plt.tight_layout()
plt.savefig(f'{E5}/gradcam_per_stage.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved figure and table to', E5)

In [ ]:
# 15. Region analysis of Grad-CAM (uses the SAVED models - no training, ~3-5 minutes)
#     Needs the functions from Cell 14 (if Colab restarted: run cells 1-6, then Cell 14 - it loads the saved models quickly)
from scipy.stats import wilcoxon

def regions(sl):
    mask = brain_mask(sl)
    ero = ndi.binary_erosion(mask, iterations=4)
    dil = ndi.binary_dilation(mask, iterations=4)
    csf = ero & (sl < 0.25)                     # dark fluid inside the brain: ventricles / deep sulci
    return {'core': ero & ~csf, 'internal_CSF': csf, 'edge_band': dil & ~ero, 'far_background': ~dil}

reg_names = ['core', 'internal_CSF', 'edge_band', 'far_background']
out_rows = []
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
for k, (tr, te) in enumerate(skf.split(np.zeros(len(stage)), stage)):
    tr, va = train_test_split(tr, test_size=0.15, stratify=stage[tr], random_state=SEED)
    m = keras.models.load_model(f'{E5}/model_fold{k}.keras')
    t1, t2 = tune(subject_score(m, va), stage[va], mf1)
    pred = apply_cuts(subject_score(m, te), t1, t2)
    conv = [l for l in m.layers if isinstance(l, layers.ReLU) and len(l.output.shape) == 4][-1]
    grad_model = keras.Model(m.inputs, [conv.output, m.output])
    for i, sidx in enumerate(te):
        cam = gradcam_subject(grad_model, sidx, 1 if pred[i] > 0 else -1)
        fr = {r: [] for r in reg_names}; right = []
        for j, c in enumerate(cam):
            tot = c.sum() + 1e-8
            for r, msk in regions(X[sidx, j].astype('float32')).items():
                fr[r].append((c * msk).sum() / tot)
            right.append(c[:, IMG // 2:].sum() / tot)          # share of attention in the right half of the image
        out_rows.append({'ID': ids[sidx], 'true': int(stage[sidx]), 'pred': int(pred[i]),
                         **{r: float(np.mean(v)) for r, v in fr.items()}, 'right_share': float(np.mean(right))})
    print(f'fold {k} done')

reg = pd.DataFrame(out_rows)
reg.to_csv(f'{E5}/gradcam_regions.csv', index=False)
pd.set_option('display.width', 200)

print('\n===== Share of Grad-CAM attention per region, by PREDICTED stage (mean) =====')
t = reg.groupby('pred')[reg_names + ['right_share']].mean().round(3); t.index = [names[i] for i in t.index]
print(t)

ok = reg[reg.true == reg.pred]
print('\n===== Same, CORRECTLY classified subjects only =====')
t = ok.groupby('true')[reg_names + ['right_share']].mean().round(3); t.index = [names[i] for i in t.index]
print(t)

print('\n===== Left-right balance (0.5 = symmetric), by predicted stage =====')
for s in range(N_CLS):
    v = reg.right_share[reg.pred == s]
    if len(v) > 5:
        print(f'{names[s]}: right share {v.mean():.3f} +/- {v.std():.3f}, Wilcoxon vs 0.5: p = {wilcoxon(v - 0.5).pvalue:.4f}')

In [ ]:
# 16. Experiment 6: augmentation and class-imbalance strategies (ordinal Custom CNN, cut-offs tuned for macro F1)
#     A = no augmentation + class weights | B = geometric + class weights (Experiment 3, reused)
#     C = geometric + intensity + class weights | D = geometric + intensity + oversampling of minority subjects
from sklearn.metrics import f1_score, cohen_kappa_score, mean_absolute_error, confusion_matrix
E6 = f'{OUT}/exp6_augmentation'; os.makedirs(E6, exist_ok=True)
NS = X.shape[1]
mf1 = lambda y, p: f1_score(y, p, average='macro', zero_division=0)
to_ord = lambda y: np.stack([y > 0, y > 1], axis=1).astype('float32')

def apply_cuts(s, t1, t2):
    return np.where(s < t1, 0, np.where(s < t2, 1, 2))

def tune(s, y, metric=mf1):
    grid, best = np.arange(0.10, 1.95, 0.05), (0.5, 1.5, -1)
    for t1 in grid:
        for t2 in grid[grid > t1]:
            v = metric(y, apply_cuts(s, t1, t2))
            if v > best[2]: best = (t1, t2, v)
    return best[:2]

def build_cnn_ordinal_aug(geometric, intensity):
    inp = keras.Input((IMG, IMG, 1)); x = inp
    if geometric:                                         # all augmentation layers are active only in training
        x = layers.RandomFlip('horizontal')(x); x = layers.RandomRotation(0.03)(x); x = layers.RandomTranslation(0.05, 0.05)(x)
    if intensity:
        x = layers.RandomBrightness(0.1, value_range=(0.0, 1.0))(x); x = layers.RandomContrast(0.2)(x)
        x = layers.GaussianNoise(0.02)(x)
    for f in (32, 64, 128):
        x = layers.Conv2D(f, 3, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x); x = layers.Dropout(0.1)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, use_bias=False, kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x); x = layers.Dropout(0.5)(x)
    out = layers.Dense(N_CLS - 1, activation='sigmoid')(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(3e-4, clipnorm=1.0), loss='binary_crossentropy')
    return m

def oversample(idx):
    """Duplicate TRAINING subjects of smaller stages until all stages are equal (never touches test subjects)."""
    rng = np.random.default_rng(SEED); target = np.bincount(stage[idx], minlength=N_CLS).max(); out = []
    for c in range(N_CLS):
        ic = idx[stage[idx] == c]; out.append(ic)
        if target > len(ic): out.append(rng.choice(ic, target - len(ic), replace=True))
    return np.concatenate(out)

def subject_score(model, idx):
    P = model.predict(X[idx].reshape(-1, IMG, IMG)[..., None].astype('float32'), batch_size=256, verbose=0)
    return P.reshape(len(idx), NS, N_CLS - 1).sum(axis=2).mean(axis=1)

VARS = {'A_none_weights':      dict(geometric=False, intensity=False, oversample=False),
        'C_geo_int_weights':   dict(geometric=True,  intensity=True,  oversample=False),
        'D_geo_int_oversample': dict(geometric=True, intensity=True,  oversample=True)}

skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)       # same subject folds as Paper 1
FOLDS = []
for tr, te in skf.split(np.zeros(len(stage)), stage):
    tr, va = train_test_split(tr, test_size=0.15, stratify=stage[tr], random_state=SEED); FOLDS.append((tr, va, te))

for name, cfg in VARS.items():
    for k, (tr, va, te) in enumerate(FOLDS):
        f = f'{E6}/{name}_fold{k}.json'
        if os.path.exists(f): print(name, 'fold', k, 'done before'); continue
        tr_use = oversample(tr) if cfg['oversample'] else tr
        Xtr, ytr = X[tr_use].reshape(-1, IMG, IMG)[..., None].astype('float32'), np.repeat(stage[tr_use], NS)
        Xva, yva = X[va].reshape(-1, IMG, IMG)[..., None].astype('float32'), np.repeat(stage[va], NS)
        cw = compute_class_weight('balanced', classes=np.arange(N_CLS), y=ytr)   # ~1 for every class after oversampling
        keras.backend.clear_session(); tf.random.set_seed(SEED)
        m = build_cnn_ordinal_aug(cfg['geometric'], cfg['intensity'])
        cb = [keras.callbacks.EarlyStopping('val_loss', patience=10, restore_best_weights=True),
              keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=4, min_lr=1e-6)]
        m.fit(Xtr, to_ord(ytr), sample_weight=cw[ytr], validation_data=(Xva, to_ord(yva), cw[yva]),
              epochs=EPOCHS, batch_size=BATCH, callbacks=cb, verbose=0)
        del Xtr, Xva
        s_va, s_te = subject_score(m, va), subject_score(m, te)
        t1, t2 = tune(s_va, stage[va]); pred = apply_cuts(s_te, t1, t2)
        json.dump({'variant': name, 'fold': k, 'test_idx': te.tolist(), 'score': s_te.tolist(),
                   'val_idx': va.tolist(), 'val_score': s_va.tolist()}, open(f, 'w'))
        print(f'{name} fold {k} | cut-offs {t1:.2f}, {t2:.2f} | bal_acc {balanced_accuracy_score(stage[te], pred):.3f}')

# ---- Summary (all variants use the same rule: cut-offs tuned for macro F1 on validation subjects) ----
def pooled(paths):
    p = np.full(len(stage), -1)
    for path in paths:
        r = json.load(open(path)); t1, t2 = tune(np.array(r['val_score']), stage[r['val_idx']])
        p[r['test_idx']] = apply_cuts(np.array(r['score']), t1, t2)
    return p

preds = {'A: none + class weights':              pooled([f'{E6}/A_none_weights_fold{k}.json' for k in range(N_FOLDS)]),
         'B: geometric + class weights':         pooled([f'{OUT}/exp3_ordinal_tuned/fold{k}.json' for k in range(N_FOLDS)]),
         'C: geometric + intensity + weights':   pooled([f'{E6}/C_geo_int_weights_fold{k}.json' for k in range(N_FOLDS)]),
         'D: geometric + intensity + oversampling': pooled([f'{E6}/D_geo_int_oversample_fold{k}.json' for k in range(N_FOLDS)])}
y = stage; rng = np.random.default_rng(SEED); boot = [rng.integers(0, len(y), len(y)) for _ in range(2000)]
rows = []
for name, p in preds.items():
    cm = confusion_matrix(y, p, labels=[0, 1, 2]); rec = cm.diagonal() / cm.sum(axis=1)
    ci = np.percentile([balanced_accuracy_score(y[i], p[i]) for i in boot], [2.5, 97.5])
    rows.append({'variant': name, 'acc': accuracy_score(y, p), 'bal_acc': balanced_accuracy_score(y, p),
                 'bal_acc 95% CI': f'[{ci[0]:.3f}, {ci[1]:.3f}]', 'macro_F1': mf1(y, p),
                 'QWK': cohen_kappa_score(y, p, weights='quadratic'), 'MAE': mean_absolute_error(y, p),
                 'ND_rec': rec[0], 'VM_rec': rec[1], 'MM_rec': rec[2]})
t = pd.DataFrame(rows).round(3); t.to_csv(f'{E6}/summary.csv', index=False)
pd.set_option('display.width', 250); print('\n===== Experiment 6: augmentation and imbalance strategies =====')
print(t.to_string(index=False))

base = preds['A: none + class weights']
for name in list(preds)[1:]:
    d = np.array([balanced_accuracy_score(y[i], preds[name][i]) - balanced_accuracy_score(y[i], base[i]) for i in boot])
    print(f'{name} vs A: bal_acc difference {balanced_accuracy_score(y, preds[name]) - balanced_accuracy_score(y, base):+.3f} '
          f'[{np.percentile(d, 2.5):+.3f}, {np.percentile(d, 97.5):+.3f}], '
          f'bootstrap p = {min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())):.3f}')

In [ ]:
# 15. Region analysis of Grad-CAM (uses the SAVED models - no training, ~3-5 minutes)
#     CORRECTED: slices with an empty Grad-CAM are skipped, so region shares add up to 100%.
#     Needs the functions from Cell 14 (run cells 1-6, then Cell 14 first - it loads the saved models quickly)
from scipy.stats import wilcoxon, kruskal

def regions(sl):
    mask = brain_mask(sl)
    ero = ndi.binary_erosion(mask, iterations=4)
    dil = ndi.binary_dilation(mask, iterations=4)
    csf = ero & (sl < 0.25)                     # dark fluid inside the brain: ventricles / deep sulci
    return {'core': ero & ~csf, 'internal_CSF': csf, 'edge_band': dil & ~ero, 'far_background': ~dil}

reg_names = ['core', 'internal_CSF', 'edge_band', 'far_background']
out_rows = []
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
for k, (tr, te) in enumerate(skf.split(np.zeros(len(stage)), stage)):
    tr, va = train_test_split(tr, test_size=0.15, stratify=stage[tr], random_state=SEED)
    m = keras.models.load_model(f'{E5}/model_fold{k}.keras')
    t1, t2 = tune(subject_score(m, va), stage[va], mf1)
    pred = apply_cuts(subject_score(m, te), t1, t2)
    conv = [l for l in m.layers if isinstance(l, layers.ReLU) and len(l.output.shape) == 4][-1]
    grad_model = keras.Model(m.inputs, [conv.output, m.output])
    for i, sidx in enumerate(te):
        cam = gradcam_subject(grad_model, sidx, 1 if pred[i] > 0 else -1)
        fr = {r: [] for r in reg_names}; right = []; n_empty = 0
        for j, c in enumerate(cam):
            if c.sum() == 0:                       # skip slices with an empty heatmap
                n_empty += 1; continue
            tot = c.sum()
            for r, msk in regions(X[sidx, j].astype('float32')).items():
                fr[r].append((c * msk).sum() / tot)
            right.append(c[:, IMG // 2:].sum() / tot)          # share of attention in the right half of the image
        out_rows.append({'ID': ids[sidx], 'true': int(stage[sidx]), 'pred': int(pred[i]),
                         **{r: (float(np.mean(v)) if v else np.nan) for r, v in fr.items()},
                         'right_share': float(np.mean(right)) if right else np.nan,
                         'empty_slices': n_empty})
    print(f'fold {k} done')

reg = pd.DataFrame(out_rows)
reg.to_csv(f'{E5}/gradcam_regions_corrected.csv', index=False)
pd.set_option('display.width', 200)

print('\n===== Empty Grad-CAM slices (out of 25 per subject), by predicted stage =====')
t = reg.groupby('pred')['empty_slices'].agg(['mean', 'sum']).round(2); t.index = [names[i] for i in t.index]
print(t)

print('\n===== Share of Grad-CAM attention per region, by PREDICTED stage (mean) =====')
t = reg.groupby('pred')[reg_names + ['right_share']].mean().round(3); t.index = [names[i] for i in t.index]
print(t)

ok = reg[reg.true == reg.pred]
print('\n===== Same, CORRECTLY classified subjects only =====')
t = ok.groupby('true')[reg_names + ['right_share']].mean().round(3); t.index = [names[i] for i in t.index]
t['n'] = ok.groupby('true').size().values
print(t)

print('\n===== Internal CSF share across TRUE stages (Kruskal-Wallis) =====')
print('p = %.4g' % kruskal(*[reg.internal_CSF[reg.true == s].dropna() for s in range(N_CLS)]).pvalue)

print('\n===== Left-right balance (0.5 = symmetric), by predicted stage =====')
for s in range(N_CLS):
    v = reg.right_share[reg.pred == s].dropna()
    if len(v) > 5:
        print(f'{names[s]}: right share {v.mean():.3f} +/- {v.std():.3f}, Wilcoxon vs 0.5: p = {wilcoxon(v - 0.5).pvalue:.4f}')